In [ ]:
!pip install -qqq datasets==3.2.0 transformers==4.47.1 trl==0.14.0 peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.2 wandb==0.19.7 --progress-bar off


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

# Wandb Authorization

In [ ]:
# from google.colab import userdata
# import os
# import wandb

# # Retrieve your stored W&B API key
# wandb_key = userdata.get('wandb_key')  # This must match the key name you used to store it

# # Set the environment variable so wandb uses it
# os.environ['WANDB_API_KEY'] = wandb_key

# # Login without prompt
# wandb.login()


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

# Load the dataset

In [ ]:
dataset = load_dataset("mlabonne/smoltldr")
print(dataset)

NameError: name 'load_dataset' is not defined

# Model

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_8bit=True,           # ✅ 8-bit quantized to save GPU memory
    torch_dtype=torch.float16,
    device_map="auto"
    #attn_implementation="flash_attention_2",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
CUDA is required but not available for bitsandbytes. Please consider installing the multi-platform enabled version of bitsandbytes, which is currently a work in progress. Please check currently supported platforms and installation instructions at https://huggingface.co/docs/bitsandbytes/main/en/installation#multi-backend


RuntimeError: CUDA is required but not available for bitsandbytes. Please consider installing the multi-platform enabled version of bitsandbytes, which is currently a work in progress. Please check currently supported platforms and installation instructions at https://huggingface.co/docs/bitsandbytes/main/en/installation#multi-backend

## Load LoRA

In [ ]:
# Load LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",  # Apply to all linear layers
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

NameError: name 'model' is not defined

## Reward function

In [ ]:
# Reward function
ideal_length = 50


def reward_len(completions, **kwargs):
    return [-abs(ideal_length - len(completion)) for completion in completions]

## Training arguments

In [ ]:
# Training arguments
training_args = GRPOConfig(
    output_dir="./grpo-output",
    learning_rate=2e-5,
    per_device_train_batch_size=2,     # ✅ small batch size
    gradient_accumulation_steps=1,
    max_prompt_length=128,             # ✅ shorter context
    max_completion_length=64,
    num_generations=4,                 # ✅ less memory than 8
    num_train_epochs=1,
    optim="adamw_8bit",                # ✅ memory-efficient optimizer
    bf16=False,
    report_to=[],
    remove_unused_columns=False,
    logging_steps=1
)

## Trainer

In [ ]:
# Trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_len],
    args=training_args,
    train_dataset=dataset["train"],
)

# Train model
# wandb.init(project="GRPO")
trainer.train()

Step,Training Loss
1,-0.000000
2,0.000500
3,0.000500
4,0.000500
5,0.000500
6,0.000700
7,0.000600
8,0.000600
9,0.000500
10,0.000500
